# Digitising in Python

This notebook uses **trace-digitiser** directly, as you would on your own computer (`pip install trace-digitiser`).
It is the same package that the **Digitise** tab and the **Terminal** run. Run the cells in order from the top: click
a cell and press **Shift+Enter**. Python is already running in your browser if you have opened the Digitise tab;
otherwise the first cell starts it.

The examples are the figures of Hearn and Whitmore (2026), in `/data/examples/hearn-whitmore-2026`. Beside each
figure is `FIGURE.known.csv`, the values it was drawn from, so every result here can be checked. Nothing you do
here leaves the browser.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import trace_digitiser as td

EX = "/data/examples/hearn-whitmore-2026"
OUT = os.path.expanduser("~/results/notebook")   # (Python does not expand ~ by itself)
os.makedirs(OUT, exist_ok=True)
print("trace-digitiser", td.__version__)

## Lines: Figure 3a

`td.digitise` reads the lines of a figure. Figure 3a has two panels, one above the other, and only the lower one
has numbers on its x axis: trace-digitiser finds both panels, reads the axes it can, and gives the upper panel the
lower one's x scale. **Read `result.warnings`** every time: they say what it was unsure of.

In [ ]:
result = td.digitise(f"{EX}/figure-3a.png", output_dir=OUT)
for message in result.warnings:
    print("CHECK:", message)
lines = result.trace_data
lines.head()

`trace_data` has a row for each column of pixels of each line: `x_value` (in the units of the x axis),
`center_estimate` (the middle of the line in that column, in the units of the y axis), `upper_envelope` and
`lower_envelope` (its top and bottom edges), and `interpolated` (True where nothing of the line was in that column).
Here are the two lines, drawn again from those values:

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(7, 5), sharex=True)
for ax, (panel, g) in zip(axes, lines.groupby("panel_id")):
    ax.plot(g["x_value"], g["center_estimate"])
    ax.set_ylabel("amplitude")
    ax.set_title(f"panel {panel}", fontsize=10)
axes[-1].set_xlabel("period (months)")
plt.show()

How close is it? The known values are the points the lines were drawn through; between them the line is
straight, so the values in between are found by interpolating.

In [ ]:
known = pd.read_csv(f"{EX}/figure-3a.known.csv")
for (panel, g), era in zip(lines.groupby("panel_id"), ["1955-1974", "1976-2014"]):
    k = known[known.panel == era].sort_values("period_months")
    inside = g[(g.x_value >= 2) & (g.x_value <= 20)]
    want = np.interp(inside.x_value, k.period_months, k.amplitude)
    err = (inside.center_estimate - want).abs()
    print(f"panel {panel} ({era}): typical error {err.median():.5f}, the y axis spans 0.25")

## Points and their error bars: Figure 3c

`td.digitise_points` finds markers - of any shape - and their error bars. In Figure 3c each dot is a ten-year
window: its x is the circular standard deviation of the peak phase of births, its y the window's middle year, on an
axis that runs downwards, and the grey bars are 95% confidence intervals.

In [ ]:
points = td.digitise_points(f"{EX}/figure-3c.png", output_dir=OUT)
for message in points.warnings:
    print("CHECK:", message)
p = points.data[["y_value", "x_value", "x_error_low", "x_error_high"]].rename(
    columns={"y_value": "year", "x_value": "circular_sd", "x_error_low": "ci_low", "x_error_high": "ci_high"})
p.head()

In [ ]:
known = pd.read_csv(f"{EX}/figure-3c.known.csv")
p["year"] = p["year"].round().astype(int)
both = p.merge(known, left_on="year", right_on="window_midpoint_year", suffixes=("", "_known"))
print(len(both), "of", len(known), "windows found")
print("typical error of the circular SD:", (both.circular_sd - both.circular_sd_known).abs().median().round(5))
print("typical error of the interval ends:", (both.ci_low - both.ci_low_known).abs().median().round(5),
      (both.ci_high - both.ci_high_known).abs().median().round(5))

## What trace-digitiser sees

`td.describe_figure` says what the automatic steps find, without digitising anything: the plot areas, the tick
labels beside each axis and what was read from them, the scale they give, and the colours drawn. The Digitise tab
shows the same over the picture. It is the place to start when a figure is not read as you expected.

In [ ]:
info = td.describe_figure(f"{EX}/figure-1a.png")
for p in info["panels"]:
    print("panel", p["panel_id"], "at", p["plot_area"])
    for axis in ("y_axis", "y2_axis", "x_axis"):
        a = p.get(axis)
        if a:
            read = [lab["text"] for lab in a["labels"]]
            print(f"  {axis}: read {read}; scale: {a.get('scale')} ({a.get('confidence')})")
    print("  colours:", [c["hex"] for c in p["colours"]])

The right-hand axis of Figure 1a is labelled with months - *June* to *January* - which are not numbers, so
no scale can be read from them. Tell trace-digitiser what two of its ticks stand for instead (`y_ref`: the row of
the tick and its value; June = 6, January = 13), and which colour to read (the orange line):

In [ ]:
orange = td.digitise(f"{EX}/figure-1a.png", trace_colours=["#ff8c00"], exclude=[(990, 15, 1296, 58)],
                     y_ref=[(485.5, 6), (34.5, 13)], output_dir=OUT, output_prefix="figure-1a-orange")
o = orange.trace_data
known = pd.read_csv(f"{EX}/figure-1a.known.csv")
want = np.interp(o.x_value, known.decimal_year, known.peak_conception_month)
print("typical error:", (o.center_estimate - want).abs().median().round(3), "months")

## Points picked by hand

`td.pixels_to_values` turns positions in the picture (pixels, from the top-left corner) into values, from two or more
ticks of each axis and the values they stand for. It is what to use for a point that nothing else can find.

In [ ]:
# two ticks of each axis of Figure 3c: x 0.1 at column 154 and 0.7 at column 533; y 1960 at row 42 and 2010 at row 683
# (the Digitise tab's status bar shows the pixel under the mouse)
td.pixels_to_values([(330, 210)], x_ref=[(154, 0.1), (533, 0.7)], y_ref=[(42, 1960), (683, 2010)])

## On your own computer

```
pip install trace-digitiser
```

and install Tesseract, which reads the tick labels (Ubuntu: `sudo apt install tesseract-ocr`; macOS:
`brew install tesseract`; Windows: the installer from the University of Mannheim). Then the code above works as it
is, with the paths of your own pictures. The help's *On your own computer* says more.